<a href="https://colab.research.google.com/github/labanaprince72-a11y/internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labanaprince72-a11y/internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

I chose **Refresh / Content Opportunity Scoring** and frame it as a **ranking/scoring** task. The decision is which content page an editor should review first. The output would be a priority score and reason codes for a review queue, not an automatic publish-or-delete decision. Ranking fits better than a yes/no classification at this stage because the practical question is "which items first?" and the editor has limited review capacity.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/labanaprince72-a11y/internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
lane = "Refresh / Content Opportunity Scoring"

print("Provisional lane:", lane)
print("Rows and columns:", df.shape)
print("One row represents a pseudonymized content item.")


Provisional lane: Refresh / Content Opportunity Scoring
Rows and columns: (30000, 44)
One row represents a pseudonymized content item.


## 2. Target or proxy

For a later model, the target should be an **observed outcome after the content action**, such as measurable recovery in a later time window. This starter snapshot does not contain a post-refresh outcome, so it cannot support a genuine future-outcome label yet. I use `refresh_opportunity_proxy` only to make the framing concrete: it combines current evidence of decline, visibility, low CTR, and staleness. It is a decision-support proxy, not a training label and not proof that a refresh will work.

In [2]:
# The starter data has no post-refresh outcome, so this is explicitly a proxy.
df["refresh_opportunity_proxy"] = (
    df["trend_direction"].eq("down").astype(int)
    + ((df["impressions_90d"] >= 100) & (df["sessions_90d"] >= 5)).astype(int)
    + (df["ctr"].fillna(0) < 0.5).astype(int)
    + (df["days_since_last_update"].fillna(0) >= 90).astype(int)
)

print("Proxy score range:", int(df["refresh_opportunity_proxy"].min()), "to", int(df["refresh_opportunity_proxy"].max()))
print("Proxy score counts:")
print(df["refresh_opportunity_proxy"].value_counts().sort_index())


Proxy score range: 0 to 4
Proxy score counts:
refresh_opportunity_proxy
0      637
1     6575
2    10657
3     8621
4     3510
Name: count, dtype: int64


## 3. Success metric

The primary metric for a later observed-label version would be **precision@K**: among the first K pages in the editor queue, how many later show the defined recovery outcome. I would choose K to match weekly editorial capacity, for example K=50, and compare the ranking with a simple baseline. A good result would improve precision@K without hiding the base rate or the cost of missed opportunities. In this notebook I only calculate a proxy sanity check because the later recovery label is not available.

In [3]:
k = min(50, len(df))
top_k = df.nlargest(k, "refresh_opportunity_proxy")
proxy_precision_at_k = (top_k["refresh_opportunity_proxy"] >= 3).mean()

print("Illustrative proxy check only — not a future-outcome metric")
print("K:", k)
print("Share of top-K items with proxy score >= 3:", round(float(proxy_precision_at_k), 3))
print("A real precision@K requires an observed outcome from a later time window.")


Illustrative proxy check only — not a future-outcome metric
K: 50
Share of top-K items with proxy score >= 3: 1.0
A real precision@K requires an observed outcome from a later time window.


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one pseudonymized content item**. Each row represents one page-level snapshot with trailing-90-day performance and content attributes. The slice below keeps the columns needed to explain the queue. I exclude IDs from features and do not use `trend_pct` or `trend_direction` as model features for a future model because they are already derived from the observed trend calculation; here they are used only to describe the current proxy.

In [4]:
lane_df = df.copy()

show_cols = [
    "content_id", "content_type", "impressions_90d", "sessions_90d",
    "ctr", "days_since_last_update", "trend_direction",
    "refresh_opportunity_proxy"
]

print("Rows in lane slice:", len(lane_df))
print("Unit of analysis: one row = one pseudonymized content item")
display(lane_df[show_cols].head(10))


Rows in lane slice: 30000
Unit of analysis: one row = one pseudonymized content item


,content_id,content_type,impressions_90d,sessions_90d,ctr,days_since_last_update,trend_direction,refresh_opportunity_proxy
0,content_304f48230142,keyword article,3803,17,0.76,20,down,2
1,content_a1fb4e703a9e,keyword article,15320,9,0.05,25,down,3
2,content_9aa793d4d895,keyword article,12581,11,0.09,20,down,3
3,content_331d6c4de07b,keyword article,11751,78,0.49,22,stable,2
4,content_d99b7a2d90ca,keyword article,19140,145,0.13,14,down,3
5,content_d4084a4bc775,keyword article,3970,5,0.03,20,down,3
6,content_9a34b442b552,keyword article,20,1,0.00,20,down,2
7,content_a63219c6e95a,keyword article,1724,28,0.06,22,stable,2
8,content_5e6c160719bc,keyword article,32574,68,0.09,20,down,3
9,content_c27558df2b0c,keyword article,1240,3,0.16,104,down,3


## 5. Why ML beats a fixed rule here

A fixed rule is a useful baseline and may be sufficient for a first version. ML would earn its place only if it improves the ranking on an observed future outcome. The pattern may be too messy for one hand-written rule because pages differ in visibility, CTR, traffic, freshness, intent, position, and engagement, and those signals can interact. A model could learn combinations and produce a calibrated priority score, but this notebook does not claim that ML is better yet. The next step would be to define a later outcome window, prevent leakage, compare a transparent rule with a model, and evaluate precision@K.

In [5]:
candidate_features = [
    "impressions_90d", "sessions_90d", "ctr", "engagement_rate",
    "avg_position", "days_since_last_update", "content_age_days",
    "search_volume", "competition"
]
feature_check = pd.DataFrame({
    "feature": candidate_features,
    "missing_values": [int(df[c].isna().sum()) for c in candidate_features],
})

print("Candidate signals for a future model (derived trend fields excluded):")
display(feature_check)
print("No model is trained here; the next step is to compare a fixed-rule baseline with an observed-label model.")


Candidate signals for a future model (derived trend fields excluded):


,feature,missing_values
0,impressions_90d,0
1,sessions_90d,0
2,ctr,0
3,engagement_rate,0
4,avg_position,0
5,days_since_last_update,0
6,content_age_days,0
7,search_volume,2468
8,competition,2468


No model is trained here; the next step is to compare a fixed-rule baseline with an observed-label model.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
